In [ ]:
!nvidia-smi

In [ ]:
%cd ..

## ✅ Part 1: Configuration and Setup

In this step we are simply importing the libraries needed and setting up a small configuration for regression training.

In [ ]:
# third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import tqdm
from diffusers import StableDiffusion3Img2ImgPipeline
from skimage.metrics import peak_signal_noise_ratio

# first-party
from fmplug.tasks.utils import prepare_super_resolution_measurement
from fmplug.utils.measurements import get_noise, get_operator

# Configuration
image_size = 512
scale_factor = 4
epochs = 10000
dtype = torch.float32

# Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 🖼️ Part 2: Load Image & Generate Measurement

In this step we have a function called `prepare_super_resolution_measurement`. One of the main outputs we need for this exercise is the original image, which is under `img_outputs["ref_img"]`.

In [ ]:
from PIL import Image

img = Image.open("/users/5/dever120/FMPlug/data/afhq_cat.png").convert("RGB")
np.array(img).shape

In [ ]:
# Load an image and simulate the measurement process
noise_sigma = 0.03
img_outputs = prepare_super_resolution_measurement(
    image_path="/users/5/dever120/FMPlug/data/afhq_cat.png",
    image_size=image_size,
    scale_factor=scale_factor,
    noise_sigma=noise_sigma,
    device=device,
    get_operator_fn=get_operator,
    get_noise_fn=get_noise,
)


## 🤖 Part 3: Load Stable Diffusion 3 Model

This is the pipeline that I have been using called `StableDiffusion3Img2ImgPipeline`. The idea is to take an image and have it transformed into a new image. This means we can start from random images or degradated images that we have available to us for inverse problems. Documentation on this pipeline can be found here: https://huggingface.co/blog/sd3#image-to-image.

For this exercise we will only extract the VAE (actually AE) encoded-decoder component. We set `.requires_grad = False` because during training we do not need the parameters of the AE to update.

In [ ]:
# Load the pretrained Stable Diffusion 3 image-to-image pipeline
print("Load in SD3 image to image model ...")
pipe = StableDiffusion3Img2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    torch_dtype=dtype,
).to(device)


vae = pipe.vae
vae.eval()
vae.requires_grad_(False)

## 📊 Part 4: Prepare Random Input & Encode with VAE

This is where we can encode our random image. There are actually two ways to do this:

1. We can use the `vae.encode` function which expects inputs of [-1, 1] that is specified in the `Img2Img` pipeline. 
2. We could also just use `torch.randn` explicity. We can do this because in the `Img2Img` pipeline when we are starting from pure noise the pipeline will actually just call `torch.randn`.

In [ ]:
psnr_scores_list = []
mse_scores_list = []

# Get the reference image
original_img = img_outputs["ref_img"]
ref_img = original_img.squeeze(0).permute(1, 2, 0)
ref_img = ref_img + 0.03 * torch.randn(ref_img.shape, device=device, dtype=torch.float)
ref_img = torch.clamp((ref_img + 1.0) / 2.0, 0.0, 1.0)
ref_img = ref_img.to(device)

# # Generate a random image and normalize
# rand_img = torch.randn(original_img.shape, device=device, dtype=dtype)
# rand_img = (rand_img - rand_img.min()) / (rand_img.max() - rand_img.min())
# rand_img = (rand_img * 2.0) - 1.0

# Noisy GT
noisy_img = original_img + 0.03 * torch.randn(original_img.shape, device=device)

# Encode using the VAE
encoded_img = vae.encode(noisy_img).latent_dist.sample()

# Histogram of encoded latent vector
ax = pd.Series(encoded_img.flatten().cpu().numpy()).hist(bins=100, figsize=(5, 4))
ax.set_title("Encoded Image Histogram")
plt.show()

In [ ]:
ref_img.min(), ref_img.max()

## 🧠 Part 5: Optimize the Latent Code

In this step we set `z = encoded_img` from part 4. `z` is a trainable parameter and will be the only parameter that updates during the regression training. For regression training we use the original (reference) image with no noise or degradation. 

During training we pass `z` through the decoder to get an image of size `(3, 512, 512)` (image size defined in the config) and take the MSE against the original image.

The metric that we track here PSNR.

In [ ]:
# Goal: Optimize the latent code z to match decoded image to reference
z = torch.randn(encoded_img.shape, device=device, dtype=torch.float32)
z = z.requires_grad_(True)

# z = encoded_img.requires_grad_(True)

# z = torch.nn.Parameter(encoded_img.clone())
optimizer = torch.optim.Adam([z], lr=1e-2)

for epoch in tqdm.tqdm(range(epochs)):
    optimizer.zero_grad()

    decoded_img = vae.decode(z).sample
    decoded_img = decoded_img.squeeze(0).permute(1, 2, 0)
    decoded_img = torch.clamp((decoded_img + 1.0) / 2.0, 0, 1)

    loss = ((ref_img - decoded_img) ** 2).mean()

    loss.backward()
    optimizer.step()

    with torch.no_grad():
        psnr_score = peak_signal_noise_ratio(
            ref_img.cpu().numpy(), decoded_img.detach().cpu().numpy()
        )
        psnr_scores_list.append(psnr_score)
        mse_scores_list.append(loss.item())

print(np.max(psnr_scores_list))

## 🎨 Part 6: Visualize & Compare Results

In this part we show the inputs and outputs of the regression model with decoder only.

1. The original image and its histogram
2. We show the random image input and its histogram
3. We show the histogram of the encoded values - this helps us get a sense of what distributions looks like in the encoded space
4. We show the trained decoded image and the histogram of the trained decoded image.

We can compare the distributions of (1) and (4) and we see that the trained distribution is more smooth than the original. From the previous cell we also get a max PSNR score.

In [ ]:
# Histogram of the original image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(ref_img.detach().cpu().numpy())

pd.Series(ref_img.detach().cpu().numpy().flatten()).hist(bins=100, ax=ax2)
ax2.set_title("Original Image Histogram")

# Decode the optimized latent vector
trained_decoded_img = vae.decode(z).sample
trained_decoded_img = (
    trained_decoded_img.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
)
trained_decoded_img = (trained_decoded_img + 1.0) / 2.0

# # Original random input image
# original_rand_img = (rand_img.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) / 2.0

# # Compare visually
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ax1.imshow(original_rand_img)
# ax1.set_title("Random Image")
# pd.Series(original_rand_img.flatten()).hist(bins=100, ax=ax2)
# ax2.set_title("Random Image Histogram")
# plt.show()

# Histogram of encoded latent vector
ax = pd.Series(encoded_img.flatten().cpu().numpy()).hist(bins=100, figsize=(5, 4))
ax.set_title("Encoded Image Histogram")
plt.show()

# Trained decoded image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(trained_decoded_img)
ax1.set_title("Trained Decoded Image")
pd.Series(trained_decoded_img.flatten()).hist(bins=100, ax=ax2)
ax2.set_title("Trained Decoded Image Histogram")
plt.show()

In [ ]:
pd.Series(psnr_scores_list).plot()

# ⚡️ Part 7: Encoder - Decoder Only

1. GT - Encoder - Decoder - Output
2. Noisy GT - Encoder - Decoder - Output

In [ ]:
original_img = img_outputs["ref_img"]
original_img = original_img + 0.03 * torch.randn(original_img.shape, device=device)

encoded_img = vae.encode(original_img).latent_dist.sample() 
decoded_img = vae.decode(encoded_img).sample

In [ ]:
# Get the PSNR for the decoder - encoder
original_img = original_img.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
decoded_img = decoded_img.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()

original_img = np.clip((original_img + 1.0) / 2.0, 0.0, 1.0)
decoded_img = np.clip((decoded_img + 1.0) / 2.0, 0.0, 1.0)

print(
    peak_signal_noise_ratio(original_img, decoded_img)
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(decoded_img)
ax1.set_title("Decoded Image")

pd.Series(decoded_img.flatten()).hist(bins=100, ax=ax2)
ax2.set_title("Original Image Histogram")

In [ ]:
pd.Series(encoded_img.detach().cpu().numpy().flatten()).hist(bins=100)
ax2.set_title("Encoded Image Histogram")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(decoded_img)
ax1.set_title("Decoded Image")

pd.Series(decoded_img.flatten()).hist(bins=100, ax=ax2)
ax2.set_title("Original Image Histogram")